In [ ]:
import scipy.io as sio
import os, glob, torch, numpy as np, json, random
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
DATA_ROOT = r'D:\mis\data'
PHANTOM_DATA_DIR = os.path.join(DATA_ROOT, 'phantom')
mat_files = sorted(glob.glob(os.path.join(PHANTOM_DATA_DIR, '**/*.mat'), recursive=True))
if not mat_files:
    raise RuntimeError(f'No phantom data found in {PHANTOM_DATA_DIR}.')

random.seed(42)
random.shuffle(mat_files)
split_idx = int(0.8 * len(mat_files))
train_files = mat_files[:split_idx]
test_files = mat_files[split_idx:]
with open(os.path.join(DATA_ROOT, 'processed', 'test_files.json'), 'w') as f:
    json.dump(test_files, f)
print(f'Train: {len(train_files)} | Test: {len(test_files)} saved to processed/test_files.json')


In [ ]:
# ── Hyperparameters & Paths ──────────────────────────────────────────────────
FT_HPARAMS = {
    'cnn_channels': [1, 16, 32, 64, 128],
    'lstm_hidden': 256,
    'lstm_layers': 2,
    'n_classes': 3,
    'patch_size': 64,
    'n_frames': 8,
    'batch_size': 8,
    'freeze_cnn_layers': 2,
    'admm_rho': 0.1,
    'tv_lambda': 0.01,
    'n_epochs': 5,
    'lr_finetune': 1e-4,
    'momentum': 0.9,
    'weight_decay': 1e-4,
    'admm_inner': 3,
    'lncc_win': 7
}
DATA_ROOT = r'D:\mis\data'
PRETRAIN_CKPT = os.path.join(DATA_ROOT, 'processed', 'cnn_lstm_pretrained_best.pth')
FINAL_CKPT    = os.path.join(DATA_ROOT, 'processed', 'cnn_lstm_final.pth')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# 🔧 Phase 2B: Unsupervised Domain Adaptation — ADMM Fine-Tuning on Phantom Data

**Dataset Used:** Dataset 2 — `ultrasound-elastography-dataset-for-unsupervised-training` (In-Vivo & Physical Phantom Data, **Train Split 80%**)

**Objective:** Adapt the pre-trained CNN-LSTM from Phase 2A to real-world ultrasound transducer artifacts **without any class labels** (fully unsupervised). The model learns to produce physically consistent strain maps on noisy phantom data.

**Pipeline:**
1. Load pre-trained weights from `cnn_lstm_pretrained_best.pth` (Phase 2A output)
2. Build DataLoader for Dataset 2 (phantom/in-vivo `.mat` files) — **no labels used**
3. **Freeze** early CNN encoder layers (protect physics knowledge)
4. **ADMM Optimization Loop:**
   - Sub-problem A (θ-step): Update network weights via SGD on smooth LNCC loss
   - Sub-problem B (z-step): Apply Proximal Soft-Thresholding to enforce TV on displacement tensors
   - Dual variable update (u-step): Update ADMM Lagrange multipliers
5. Monitor unsupervised LNCC + TV convergence across epochs
6. Save final production weights → `cnn_lstm_final.pth`

> **Integrates Syllabus Unit 2:** Matrix Splitting (ADMM), Proximal Algorithms (Soft-Thresholding), Total Variation Regularization, SGD

## 0. Environment Setup & Imports

In [ ]:
# Redundant or synthetic generation removed by standardization script


In [ ]:
class PhantomDataset(Dataset):
    def __init__(self, file_list, patch_size=64, n_frames=10):
        self.files = file_list
        self.patch_size = patch_size
        self.n_frames = n_frames
    def __len__(self):
        return len(self.files)
    def __getitem__(self, idx):
        rf_pre, rf_post = extract_rf_pair(self.files[idx])
        d, w, t = rf_pre.shape
        s_d, s_w = (d - self.patch_size)//2, (w - self.patch_size)//2
        p_pre = rf_pre[s_d:s_d+self.patch_size, s_w:s_w+self.patch_size, :self.n_frames]
        p_post = rf_post[s_d:s_d+self.patch_size, s_w:s_w+self.patch_size, :self.n_frames]
        return torch.from_numpy(p_pre).permute(2,0,1).unsqueeze(1).float(), \
               torch.from_numpy(p_post).permute(2,0,1).unsqueeze(1).float()


## 1. Re-instantiate CNN-LSTM Architecture (identical to Phase 2A)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  Architecture definitions must match Phase 2A exactly.
#  In a production codebase these would be imported from ml_engine/architecture.py
# ─────────────────────────────────────────────────────────────────────────────

class ConvBlock(nn.Module):
    """Double convolution block with BatchNorm and GELU activation."""
    def __init__(self, in_ch: int, out_ch: int, dropout: float = 0.1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
            nn.Dropout2d(dropout)
        )
    def forward(self, x):
        return self.block(x)


class UNetEncoder(nn.Module):
    def __init__(self, channels: list = [1, 16, 32, 64, 128]):
        super().__init__()
        self.encoders = nn.ModuleList()
        self.pools    = nn.ModuleList()
        for i in range(len(channels) - 1):
            self.encoders.append(ConvBlock(channels[i], channels[i+1]))
            if i < len(channels) - 2:
                self.pools.append(nn.MaxPool2d(2, 2))
        self.feat_dim    = channels[-1]
        self.global_pool = nn.AdaptiveAvgPool2d(1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for i, enc in enumerate(self.encoders):
            x = enc(x)
            if i < len(self.pools):
                x = self.pools[i](x)
        return self.global_pool(x).squeeze(-1).squeeze(-1)


class CNNLSTMElastography(nn.Module):
    """
    Physics-informed CNN-LSTM for ultrasound elastography tumor grading.
    Input:  (B, T, 1, H, W)
    Output: logits (B, n_classes), displacement (B, T, H, W)
    """
    def __init__(
        self,
        cnn_channels : list = [1, 16, 32, 64, 128],
        lstm_hidden  : int  = 256,
        lstm_layers  : int  = 2,
        n_classes    : int  = 3,
        patch_size   : int  = 64,
        dropout      : float = 0.3
    ):
        super().__init__()
        self.patch_size  = patch_size
        self.lstm_hidden = lstm_hidden

        self.cnn   = UNetEncoder(channels=cnn_channels)
        feat_dim   = cnn_channels[-1]

        self.lstm  = nn.LSTM(
            input_size   = feat_dim,
            hidden_size  = lstm_hidden,
            num_layers   = lstm_layers,
            batch_first  = True,
            bidirectional= True,
            dropout      = dropout if lstm_layers > 1 else 0.0
        )
        lstm_out_dim = lstm_hidden * 2

        self.classifier = nn.Sequential(
            nn.LayerNorm(lstm_out_dim),
            nn.Linear(lstm_out_dim, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(64, n_classes)
        )

        reduced = patch_size // (2 ** (len(cnn_channels) - 2))
        reduced = max(reduced, 4)
        self.disp_head = nn.Sequential(
            nn.Linear(lstm_out_dim, 256),
            nn.GELU(),
            nn.Linear(256, reduced * reduced)
        )
        self.disp_upsample = nn.Upsample(
            size=(patch_size, patch_size), mode='bilinear', align_corners=False
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, (nn.BatchNorm2d, nn.LayerNorm)):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor):
        B, T, C, H, W = x.shape
        x_flat  = x.view(B * T, C, H, W)
        feats   = self.cnn(x_flat).view(B, T, -1)
        lstm_out, (h_n, _) = self.lstm(feats)
        h_final = torch.cat([h_n[-2], h_n[-1]], dim=-1)
        logits  = self.classifier(h_final)

        disp_flat    = self.disp_head(lstm_out.reshape(B * T, -1))
        r            = int(disp_flat.shape[-1] ** 0.5)
        disp_spatial = disp_flat.view(B * T, 1, r, r)
        disp_up      = self.disp_upsample(disp_spatial)
        displacement = disp_up.view(B, T, H, W)
        return logits, displacement


print('✅ Architecture defined')

## 2. Load Pre-trained Weights & Apply Layer Freezing

In [ ]:
# ── Instantiate model ──────────────────────────────────────────────────────────
model = CNNLSTMElastography(
    cnn_channels = FT_HPARAMS['cnn_channels'],
    lstm_hidden  = FT_HPARAMS['lstm_hidden'],
    lstm_layers  = FT_HPARAMS['lstm_layers'],
    n_classes    = FT_HPARAMS['n_classes'],
    patch_size   = FT_HPARAMS['patch_size']
).to(DEVICE)

# ── Load pre-trained weights ───────────────────────────────────────────────────
if os.path.isfile(PRETRAIN_CKPT):
    checkpoint = torch.load(PRETRAIN_CKPT, map_location=DEVICE)
    state_dict = checkpoint.get('state_dict', checkpoint)  # handle both formats
    model.load_state_dict(state_dict, strict=True)
    loaded_epoch = checkpoint.get('epoch', 'unknown')
    loaded_val   = checkpoint.get('val_loss', float('nan'))
    print(f'✅ Pre-trained weights loaded from: {PRETRAIN_CKPT}')
    print(f'   Checkpoint epoch: {loaded_epoch}  |  Val Loss: {loaded_val:.4f}')
else:
    print('⚠️  Pre-trained checkpoint not found — initialising from scratch (demo mode)')
    print(f'   Expected: {PRETRAIN_CKPT}')

# ── Selective Layer Freezing ───────────────────────────────────────────────────
# Strategy: Freeze the first N CNN encoder blocks (low-level features are
# already physics-grounded). Fine-tune LSTM, deeper CNN, and all heads.
n_freeze = FT_HPARAMS['freeze_cnn_layers']

frozen_params = 0
active_params = 0

for i, encoder_block in enumerate(model.cnn.encoders):
    if i < n_freeze:
        for p in encoder_block.parameters():
            p.requires_grad = False
            frozen_params += p.numel()
    else:
        for p in encoder_block.parameters():
            p.requires_grad = True
            active_params += p.numel()

# Pooling has no learnable params; LSTM, classifier, disp_head always active
for name, p in model.named_parameters():
    if p.requires_grad:
        active_params += p.numel() if 'cnn.encoders' not in name else 0

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'\n🔒 Layer Freezing Summary')
print(f'   Total parameters     : {total_params:,}')
print(f'   Frozen (first {n_freeze} CNN blocks): {total_params - trainable:,}')
print(f'   Active (trainable)   : {trainable:,}  ({100*trainable/total_params:.1f}%)')

## 3. Phantom Dataset & DataLoader (Dataset 2 — Unsupervised)

In [ ]:
def extract_rf_pair(fpath: str):
    """
    Load .mat file and extract pre/post compression RF matrices.
    Returns (rf_pre, rf_post) as (D, W, F) float32 arrays.
    """
    import scipy.io as sio
    mat_data = sio.loadmat(fpath)
    keys = list(mat_data.keys())

    # Strategy 1: explicit pre/post keys
    rf_pre, rf_post = None, None
    for pk in ['rf_pre', 'IQpre', 'pre', 'rfPre', 'RF_pre']:
        if pk in mat_data:
            rf_pre = np.array(mat_data[pk], dtype=np.float32)
            break
    for pk in ['rf_post', 'IQpost', 'post', 'rfPost', 'RF_post']:
        if pk in mat_data:
            rf_post = np.array(mat_data[pk], dtype=np.float32)
            break

    # Strategy 2: single stacked array (like 'Data' or 'RF')
    if rf_pre is None:
        for pk in ['Data', 'RF', 'rf', 'rfData', 'data', 'frames']:
            if pk in mat_data:
                arr = np.array(mat_data[pk], dtype=np.float32)
                
                # Handle (n_frames, D, W) -> (D, W, n_frames)
                if arr.ndim == 3 and arr.shape[0] < arr.shape[1] and arr.shape[0] < arr.shape[2]:
                    arr = arr.transpose(1, 2, 0)
                
                if arr.ndim == 3 and arr.shape[-1] >= 2:
                    mid = arr.shape[-1] // 2
                    rf_pre  = arr[:, :, :mid]
                    rf_post = arr[:, :, mid:]
                elif arr.ndim == 2:
                    rf_pre  = arr[:, :, np.newaxis]
                    rf_post = arr[:, :, np.newaxis]
                break

    if rf_pre is None:
        raise KeyError(f'Cannot find RF pair in {fpath}. Available keys: {keys}')

    # Ensure 3D (D, W, F)
    if rf_pre.ndim == 2:  rf_pre  = rf_pre[:, :, np.newaxis]
    if rf_post.ndim == 2: rf_post = rf_post[:, :, np.newaxis]

    return rf_pre, rf_post

In [ ]:
# ── Build DataLoader ──────────────────────────────────────────────────────────
train_dataset = PhantomDataset(
    mat_files  = train_files,
    patch_size = FT_HPARAMS['patch_size'],
    n_frames   = FT_HPARAMS['n_frames'],
    stride     = FT_HPARAMS['patch_size'] // 2
)

train_loader = DataLoader(
    train_dataset,
    batch_size  = FT_HPARAMS['batch_size'],
    shuffle     = True,
    num_workers = 0,
    pin_memory  = (DEVICE.type == 'cuda')
)

print(f'\n  DataLoader: {len(train_dataset)} patches | {len(train_loader)} batches')

# Verify batch shape
x_sample = next(iter(train_loader))
print(f'  Sample batch shape: {x_sample.shape}  → (B, T, 1, H, W)')
print(f'  Value range: [{x_sample.min():.3f}, {x_sample.max():.3f}]')

## 4. Unsupervised Loss Functions (Label-Free)

During domain adaptation we have **no class labels**. The model is supervised purely by:

| Loss | Formula | Purpose |
|------|---------|----------|
| **LNCC** | $1 - \frac{\sum_\Omega(u-\bar{u})(v-\bar{v})}{\sqrt{\sum_\Omega(u-\bar u)^2 \cdot \sum_\Omega(v-\bar v)^2}}$ | Acoustic tracking: consecutive displacement frames should correlate |
| **Spatio-Temporal TV** | $\sum_{x,y,t}\|\nabla d\|_1$ | Tissue does not deform discontinuously — enforces physical continuity |

The ADMM formulation separates these into smooth (LNCC) and non-smooth (TV) sub-problems.

In [ ]:
class LNCCLoss(nn.Module):
    """
    Local Normalized Cross-Correlation Loss.
    Self-supervised: compares consecutive predicted displacement frames.
    """
    def __init__(self, win: int = 9):
        super().__init__()
        self.win = win

    def forward(self, pred: torch.Tensor) -> torch.Tensor:
        """
        pred: (B, T, H, W) displacement field
        Returns scalar LNCC loss (lower = better acoustic tracking).
        """
        if pred.shape[1] < 2:
            return torch.tensor(0.0, device=pred.device)

        u = pred[:, :-1, :, :].contiguous()
        v = pred[:, 1:,  :, :].contiguous()
        B, T1, H, W = u.shape
        w, pad = self.win, self.win // 2

        u_f = u.reshape(B * T1, 1, H, W)
        v_f = v.reshape(B * T1, 1, H, W)

        kernel = torch.ones(1, 1, w, w, device=u.device) / (w * w)
        u_mu   = F.conv2d(u_f, kernel, padding=pad)
        v_mu   = F.conv2d(v_f, kernel, padding=pad)
        uc     = u_f - u_mu
        vc     = v_f - v_mu

        cross = F.conv2d(uc * vc, kernel, padding=pad)
        u_sq  = F.conv2d(uc * uc, kernel, padding=pad)
        v_sq  = F.conv2d(vc * vc, kernel, padding=pad)

        lncc  = cross / (torch.sqrt(u_sq * v_sq + 1e-8) + 1e-8)
        return (1.0 - lncc.clamp(-1, 1)).mean()


class SpatioTemporalTVLoss(nn.Module):
    """
    Isotropic Spatio-Temporal Total Variation regularization.
    TV(d) = Σ_{x,y,t} |∂d/∂x| + |∂d/∂y| + α|∂d/∂t|
    """
    def __init__(self, temporal_weight: float = 0.5):
        super().__init__()
        self.alpha = temporal_weight

    def forward(self, disp: torch.Tensor) -> torch.Tensor:
        """disp: (B, T, H, W)"""
        dx = torch.abs(disp[:, :, 1:, :]  - disp[:, :, :-1, :])
        dy = torch.abs(disp[:, :, :, 1:]  - disp[:, :, :, :-1])
        dt = torch.abs(disp[:, 1:, :, :]  - disp[:, :-1, :, :])
        tv_s = (dx[:, :, :, :-1] + dy[:, :, :-1, :]).mean()
        tv_t = dt.mean()
        return tv_s + self.alpha * tv_t


# ── Instantiate losses ────────────────────────────────────────────────────────
lncc_criterion = LNCCLoss(win=FT_HPARAMS['lncc_win']).to(DEVICE)
tv_criterion   = SpatioTemporalTVLoss(temporal_weight=0.5)

print('✅ Unsupervised loss functions instantiated')
print(f'   LNCC window : {FT_HPARAMS["lncc_win"]}×{FT_HPARAMS["lncc_win"]}')
print(f'   TV lambda   : {FT_HPARAMS["tv_lambda"]}')

## 5. ADMM Optimizer — Mathematical Framework

Standard SGD cannot jointly optimize the **smooth** LNCC and **non-smooth** TV loss. We use **ADMM (Alternating Direction Method of Multipliers)** with variable splitting:

**Primal problem:**
$$\min_{\theta, d} \ \mathcal{L}_{\text{LNCC}}(d) + \lambda \|d\|_{TV}$$

**ADMM variable splitting:** Introduce auxiliary variable $z \approx d$:
$$\min_{\theta, d, z} \ \mathcal{L}_{\text{LNCC}}(d) + \lambda \|z\|_{TV} \quad \text{s.t.} \ d = z$$

**Augmented Lagrangian:**
$$\mathcal{L}_\rho(\theta, d, z, u) = \mathcal{L}_{\text{LNCC}} + \lambda\|z\|_{TV} + \frac{\rho}{2}\|d - z + u\|_2^2$$

**Three-step update per iteration:**

| Step | Variable | Update Rule |
|------|----------|-------------|
| **θ-step (A)** | Network weights | $\theta \leftarrow \theta - \eta \nabla_\theta [\mathcal{L}_{\text{LNCC}} + \frac{\rho}{2}\|d - z + u\|^2]$ via SGD |
| **z-step (B)** | Auxiliary variable | $z \leftarrow \text{prox}_{\lambda/\rho \|\cdot\|_{TV}}(d + u)$ via **Soft-Thresholding** |
| **u-step** | Dual variable | $u \leftarrow u + (d - z)$ (dual ascent) |

In [ ]:
def soft_threshold(x: torch.Tensor, threshold: float) -> torch.Tensor:
    """
    Element-wise Soft-Thresholding (Proximal operator for L1 / TV).

    prox_{τ||·||_1}(x) = sign(x) * max(|x| - τ, 0)

    This is the closed-form solution to:
        argmin_z  (1/2)||z - x||^2  +  τ||z||_1

    Applied to the displacement field gradient it approximates
    the TV proximal operator (gradient-domain thresholding).

    Args:
        x         : input tensor
        threshold : τ = λ / ρ  (TV weight / ADMM penalty)
    Returns:
        z : thresholded tensor (same shape as x)
    """
    return torch.sign(x) * torch.clamp(torch.abs(x) - threshold, min=0.0)


def admm_z_step(
    d: torch.Tensor,
    u: torch.Tensor,
    threshold: float
) -> torch.Tensor:
    """
    ADMM Sub-problem B (z-step): Proximal update for TV.

    z = prox_{(λ/ρ)||·||_TV}(d + u)

    Approximation: Apply soft-thresholding on the gradient domain
    (shrinks spatial/temporal gradient magnitude).

    Args:
        d         : displacement field (B, T, H, W)
        u         : dual variable (B, T, H, W)
        threshold : τ = λ / ρ
    Returns:
        z : updated auxiliary variable
    """
    v = d + u  # consensus target

    # Compute gradients in all 3 dimensions
    # Spatial x-gradient
    gx = torch.zeros_like(v)
    gx[:, :, :-1, :] = v[:, :, 1:, :] - v[:, :, :-1, :]

    # Spatial y-gradient
    gy = torch.zeros_like(v)
    gy[:, :, :, :-1] = v[:, :, :, 1:] - v[:, :, :, :-1]

    # Temporal t-gradient
    gt = torch.zeros_like(v)
    gt[:, :-1, :, :] = v[:, 1:, :, :] - v[:, :-1, :, :]

    # Soft-threshold gradients
    gx_thresh = soft_threshold(gx, threshold)
    gy_thresh = soft_threshold(gy, threshold)
    gt_thresh = soft_threshold(gt, threshold)

    # Reconstruct from thresholded gradients (divergence operation)
    # z ≈ v - div(prox(grad(v)))  [gradient-domain shrinkage approximation]
    div_x = torch.zeros_like(v)
    div_x[:, :, :-1, :] = gx_thresh[:, :, :-1, :]
    div_x[:, :, 1:, :]  = div_x[:, :, 1:, :] - gx_thresh[:, :, :-1, :]

    div_y = torch.zeros_like(v)
    div_y[:, :, :, :-1] = gy_thresh[:, :, :, :-1]
    div_y[:, :, :, 1:]  = div_y[:, :, :, 1:] - gy_thresh[:, :, :, :-1]

    div_t = torch.zeros_like(v)
    div_t[:, :-1, :, :] = gt_thresh[:, :-1, :, :]
    div_t[:, 1:,  :, :] = div_t[:, 1:, :, :] - gt_thresh[:, :-1, :, :]

    z = v - (div_x + div_y + 0.5 * div_t)
    return z


class ADMMFineTuner:
    """
    ADMM-based fine-tuner for unsupervised domain adaptation.

    Manages three variables:
        θ : model parameters (network weights)
        z : auxiliary displacement variable (TV prox target)
        u : scaled dual variable (Lagrange multiplier)

    Per step:
        1. θ-step: SGD update on LNCC + ADMM penalty
        2. z-step: proximal soft-threshold update
        3. u-step: dual variable ascent
    """

    def __init__(
        self,
        model     : nn.Module,
        rho       : float = 0.1,
        tv_lambda : float = 0.05,
        lr        : float = 5e-5,
        momentum  : float = 0.9,
        weight_decay: float = 1e-4,
        n_inner   : int   = 3,
        device    : torch.device = torch.device('cpu')
    ):
        self.model     = model
        self.rho       = rho
        self.lam       = tv_lambda
        self.threshold = tv_lambda / rho   # τ = λ/ρ  (proximal threshold)
        self.n_inner   = n_inner
        self.device    = device

        # SGD optimizer for θ-step (only trainable params)
        trainable = [p for p in model.parameters() if p.requires_grad]
        self.sgd  = torch.optim.SGD(
            trainable, lr=lr, momentum=momentum, weight_decay=weight_decay
        )
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            self.sgd, T_max=20, eta_min=lr * 0.1
        )

        # ADMM auxiliary variables (lazily initialised per batch)
        self.z = None
        self.u = None

        self.lncc_fn = LNCCLoss(win=9).to(device)

    def _init_admm_vars(self, d: torch.Tensor):
        """Lazily initialise z and u to zero with same shape as d."""
        if self.z is None or self.z.shape != d.shape:
            self.z = torch.zeros_like(d, requires_grad=False)
            self.u = torch.zeros_like(d, requires_grad=False)

    def step(
        self,
        x_batch: torch.Tensor
    ) -> dict:
        """
        Full ADMM update for one mini-batch.

        Args:
            x_batch: (B, T, 1, H, W) RF frame sequence
        Returns:
            info dict with scalar losses for logging
        """
        total_lncc = 0.0
        total_admm_pen = 0.0
        total_tv = 0.0

        for k in range(self.n_inner):

            # ──────────────────────────────────────────────────────────────────
            # Sub-problem A: θ-step (network weight update via SGD)
            # min_θ  L_LNCC(d(θ)) + (ρ/2)||d(θ) - z + u||²
            # ──────────────────────────────────────────────────────────────────
            self.model.train()
            self.sgd.zero_grad()

            _, disp = self.model(x_batch)   # (B, T, H, W)

            # Initialise ADMM vars on first encounter
            self._init_admm_vars(disp.detach())

            # LNCC: smooth term (maximise acoustic correlation)
            loss_lncc = self.lncc_fn(disp)

            # ADMM penalty: (ρ/2)||d - z + u||²  (consensus constraint)
            penalty   = (self.rho / 2.0) * torch.norm(disp - self.z + self.u, p='fro') ** 2
            penalty   = penalty / disp.numel()   # normalise by tensor size

            loss_A = loss_lncc + penalty
            loss_A.backward()
            torch.nn.utils.clip_grad_norm_(
                [p for p in self.model.parameters() if p.requires_grad], max_norm=1.0
            )
            self.sgd.step()

            total_lncc    += loss_lncc.item()
            total_admm_pen += penalty.item()

            # ──────────────────────────────────────────────────────────────────
            # Sub-problem B: z-step (Proximal soft-thresholding)
            # min_z  λ||z||_TV  +  (ρ/2)||d - z + u||²
            # Solution: z = prox_{λ/ρ · TV}(d + u)
            # ──────────────────────────────────────────────────────────────────
            with torch.no_grad():
                d_detach  = disp.detach()
                self.z    = admm_z_step(d_detach, self.u, self.threshold)
                tv_val    = self.lam * (torch.abs(self.z[:, :, 1:, :] - self.z[:, :, :-1, :]).mean() +
                                        torch.abs(self.z[:, :, :, 1:] - self.z[:, :, :, :-1]).mean())
                total_tv  += tv_val.item()

                # ──────────────────────────────────────────────────────────────
                # Dual variable update: u = u + (d - z)
                # ──────────────────────────────────────────────────────────────
                self.u = self.u + (d_detach - self.z)

                # Dual variable clipping to prevent unbounded growth
                self.u = torch.clamp(self.u, -10.0, 10.0)

        primal_res = torch.norm(disp.detach() - self.z).item() / disp.numel()
        dual_res   = torch.norm(self.rho * (self.z - (self.z if self.z is not None else self.z))).item()

        return {
            'lncc'        : total_lncc    / self.n_inner,
            'admm_penalty': total_admm_pen / self.n_inner,
            'tv'          : total_tv       / self.n_inner,
            'total_loss'  : (total_lncc + total_admm_pen) / self.n_inner,
            'primal_res'  : primal_res,
        }


# ── Instantiate ADMM fine-tuner ────────────────────────────────────────────────
admm_tuner = ADMMFineTuner(
    model       = model,
    rho         = FT_HPARAMS['admm_rho'],
    tv_lambda   = FT_HPARAMS['tv_lambda'],
    lr          = FT_HPARAMS['lr_finetune'],
    momentum    = FT_HPARAMS['momentum'],
    weight_decay= FT_HPARAMS['weight_decay'],
    n_inner     = FT_HPARAMS['admm_inner'],
    device      = DEVICE
)

print(f'✅ ADMM Fine-Tuner ready')
print(f'   ρ (rho)      : {admm_tuner.rho}')
print(f'   λ (tv_lambda): {admm_tuner.lam}')
print(f'   τ (threshold): {admm_tuner.threshold:.4f}   [= λ/ρ]')
print(f'   Inner iters K: {admm_tuner.n_inner}')

## 6. Fine-Tuning Training Loop

In [ ]:
# ── Logging ───────────────────────────────────────────────────────────────────
ft_history = {
    'epoch'       : [],
    'lncc'        : [],
    'tv'          : [],
    'admm_penalty': [],
    'total_loss'  : [],
    'primal_res'  : [],
    'lr'          : [],
}

best_loss    = float('inf')
best_epoch   = -1
N_EPOCHS     = FT_HPARAMS['n_epochs']

print(f'\n🚀 Starting Unsupervised ADMM Fine-Tuning')
print(f'   Epochs    : {N_EPOCHS}')
print(f'   Batches   : {len(train_loader)} per epoch')
print(f'   Device    : {DEVICE}')
print('=' * 72)

for epoch in range(1, N_EPOCHS + 1):

    # ── Reset ADMM dual variables at epoch start for stability ────────────────
    if epoch % 5 == 1 and epoch > 1:
        admm_tuner.z = None
        admm_tuner.u = None
        print(f'  [Epoch {epoch}] ADMM dual variables reset for re-convergence')

    epoch_metrics = {k: 0.0 for k in ['lncc', 'tv', 'admm_penalty', 'total_loss', 'primal_res']}
    n_batches = 0

    loop = tqdm(
        train_loader,
        desc=f'Epoch {epoch:03d}/{N_EPOCHS}',
        leave=False,
        ncols=90
    )

    for x_batch in loop:
        x_batch = x_batch.to(DEVICE)   # (B, T, 1, H, W)

        step_info = admm_tuner.step(x_batch)

        for k in epoch_metrics:
            epoch_metrics[k] += step_info.get(k, 0.0)
        n_batches += 1

        loop.set_postfix(
            lncc=f'{step_info["lncc"]:.4f}',
            tv=f'{step_info["tv"]:.4f}',
            res=f'{step_info["primal_res"]:.4f}'
        )

    # ── Aggregate epoch metrics ───────────────────────────────────────────────
    n = max(n_batches, 1)
    avg = {k: v / n for k, v in epoch_metrics.items()}
    current_lr = admm_tuner.sgd.param_groups[0]['lr']

    for k in avg:
        ft_history[k].append(avg[k])
    ft_history['epoch'].append(epoch)
    ft_history['lr'].append(current_lr)

    admm_tuner.scheduler.step()

    print(
        f'Epoch {epoch:03d}/{N_EPOCHS} | '
        f'LNCC: {avg["lncc"]:.4f}  '
        f'TV: {avg["tv"]:.4f}  '
        f'Penalty: {avg["admm_penalty"]:.4f}  '
        f'Total: {avg["total_loss"]:.4f}  '
        f'Primal_res: {avg["primal_res"]:.5f}  '
        f'LR: {current_lr:.2e}'
    )

    # ── Save best model (lowest LNCC = best acoustic tracking) ───────────────
    if avg['lncc'] < best_loss:
        best_loss  = avg['lncc']
        best_epoch = epoch
        torch.save({
            'epoch'         : epoch,
            'state_dict'    : model.state_dict(),
            'lncc_loss'     : avg['lncc'],
            'tv_loss'       : avg['tv'],
            'ft_hparams'    : FT_HPARAMS,
            'frozen_layers' : FT_HPARAMS['freeze_cnn_layers'],
        }, FINAL_CKPT)
        print(f'  ✅ Best model saved → {FINAL_CKPT}')

print('\n' + '=' * 72)
print(f'🏁 Fine-tuning complete.')
print(f'   Best epoch : {best_epoch}  |  Best LNCC: {best_loss:.4f}')
print(f'   Weights    : {FINAL_CKPT}')

## 7. Convergence Diagnostics & Visualisation

In [ ]:
# ── Training curves ───────────────────────────────────────────────────────────
epochs_x = ft_history['epoch']

fig = plt.figure(figsize=(18, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# ── Panel 1: Total Loss ───────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(epochs_x, ft_history['total_loss'], 'b-o', ms=4, lw=1.5, label='Total Loss')
ax1.axvline(best_epoch, ls='--', color='green', alpha=0.7, label=f'Best (ep.{best_epoch})')
ax1.set_title('Total ADMM Loss', fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(fontsize=8); ax1.grid(True, alpha=0.3)

# ── Panel 2: LNCC ─────────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(epochs_x, ft_history['lncc'], 'g-s', ms=4, lw=1.5, label='LNCC')
ax2.set_title('LNCC Loss (Acoustic Tracking)', fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('LNCC')
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)

# ── Panel 3: TV ───────────────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(epochs_x, ft_history['tv'], 'm-^', ms=4, lw=1.5, label='TV Regularization')
ax3.set_title('TV Loss (Physical Continuity)', fontweight='bold')
ax3.set_xlabel('Epoch'); ax3.set_ylabel('TV')
ax3.legend(fontsize=8); ax3.grid(True, alpha=0.3)

# ── Panel 4: ADMM Primal Residual ─────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 0])
ax4.semilogy(epochs_x, [r + 1e-10 for r in ft_history['primal_res']], 'r-D', ms=4, lw=1.5)
ax4.set_title('ADMM Primal Residual ||d - z||', fontweight='bold')
ax4.set_xlabel('Epoch'); ax4.set_ylabel('Residual (log scale)')
ax4.grid(True, alpha=0.3, which='both')
ax4.set_ylabel('||d - z||')

# ── Panel 5: Learning Rate ────────────────────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 1])
ax5.semilogy(epochs_x, ft_history['lr'], 'c-o', ms=4, lw=1.5)
ax5.set_title('Learning Rate Schedule (Cosine)', fontweight='bold')
ax5.set_xlabel('Epoch'); ax5.set_ylabel('LR (log scale)')
ax5.grid(True, alpha=0.3, which='both')

# ── Panel 6: LNCC vs TV tradeoff scatter ──────────────────────────────────────
ax6 = fig.add_subplot(gs[1, 2])
sc  = ax6.scatter(
    ft_history['lncc'], ft_history['tv'],
    c=epochs_x, cmap='plasma', s=40, zorder=5
)
plt.colorbar(sc, ax=ax6, label='Epoch')
ax6.set_title('LNCC vs TV Tradeoff', fontweight='bold')
ax6.set_xlabel('LNCC Loss'); ax6.set_ylabel('TV Loss')
ax6.grid(True, alpha=0.3)

fig.suptitle(
    'Phase 2B: ADMM Unsupervised Fine-Tuning — Convergence Diagnostics\n'
    f'(Dataset 2: Phantom/In-Vivo | {len(train_files)} train files)',
    fontsize=13, fontweight='bold', y=1.01
)

plt.savefig(os.path.join(OUT_DIR, 'finetune_convergence.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Convergence plot saved → {OUT_DIR}finetune_convergence.png')

In [ ]:
# ── Visualise displacement fields before vs after fine-tuning ─────────────────
model.eval()

with torch.no_grad():
    demo_batch = next(iter(train_loader)).to(DEVICE)   # (B, T, 1, H, W)
    _, disp_final = model(demo_batch)

disp_np   = disp_final[0].cpu().numpy()   # (T, H, W)
T_disp    = disp_np.shape[0]
n_show    = min(4, T_disp)
frame_ids = np.linspace(0, T_disp - 1, n_show, dtype=int)

fig, axes = plt.subplots(2, n_show, figsize=(4 * n_show, 7))

for col, t in enumerate(frame_ids):
    # Top row: input RF frame
    rf_frame = demo_batch[0, t, 0].cpu().numpy()
    axes[0, col].imshow(rf_frame, cmap='gray', aspect='auto')
    axes[0, col].set_title(f'RF Frame t={t}', fontsize=9)
    axes[0, col].axis('off')

    # Bottom row: displacement field (strain map)
    im = axes[1, col].imshow(
        disp_np[t], cmap='RdBu_r', aspect='auto',
        vmin=disp_np.min(), vmax=disp_np.max()
    )
    axes[1, col].set_title(f'Strain Map t={t}', fontsize=9)
    axes[1, col].axis('off')
    plt.colorbar(im, ax=axes[1, col], shrink=0.7, label='Displacement')

axes[0, 0].set_ylabel('RF Input', fontsize=10, labelpad=12)
axes[1, 0].set_ylabel('Strain Map', fontsize=10, labelpad=12)

plt.suptitle(
    'Fine-Tuned Displacement Fields on Phantom Data\n'
    '(Output of ADMM-adapted CNN-LSTM)',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'displacement_fields_finetuned.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Displacement field visualisation saved')

In [ ]:
# ── Proximal operator demonstration ───────────────────────────────────────────
# Visualise how soft-thresholding shapes the displacement gradient distribution

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Raw displacement gradient histogram
raw_grad  = torch.abs(disp_final[0, :, 1:, :] - disp_final[0, :, :-1, :]).cpu().numpy().flatten()
axes[0].hist(raw_grad, bins=80, color='steelblue', alpha=0.7, density=True)
axes[0].set_title('Raw Displacement Gradient |∂d/∂x|', fontweight='bold')
axes[0].set_xlabel('Gradient magnitude'); axes[0].set_ylabel('Density')
axes[0].axvline(admm_tuner.threshold, color='red', ls='--', label=f'τ={admm_tuner.threshold:.3f}')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# 2. Soft-threshold function plot
x_vals  = np.linspace(-1.5, 1.5, 500)
tau     = admm_tuner.threshold
y_vals  = np.sign(x_vals) * np.maximum(np.abs(x_vals) - tau, 0)
axes[1].plot(x_vals, x_vals, 'b--', alpha=0.4, label='Identity')
axes[1].plot(x_vals, y_vals, 'r-',  lw=2.5,    label=f'Soft-Threshold (τ={tau:.3f})')
axes[1].fill_between(x_vals, x_vals, y_vals, alpha=0.15, color='red', label='Shrinkage region')
axes[1].axhline(0, color='k', lw=0.5); axes[1].axvline(0, color='k', lw=0.5)
axes[1].set_title('Proximal Operator: Soft-Thresholding', fontweight='bold')
axes[1].set_xlabel('Input x'); axes[1].set_ylabel('Output sign(x)·max(|x|−τ, 0)')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)
axes[1].set_xlim(-1.5, 1.5); axes[1].set_ylim(-1.5, 1.5)

# 3. Thresholded gradient histogram
thresh_grad = np.sign(raw_grad) * np.maximum(np.abs(raw_grad) - tau, 0)
axes[2].hist(thresh_grad, bins=80, color='darkorange', alpha=0.7, density=True)
axes[2].set_title('Thresholded Gradient (Post Proximal Step)', fontweight='bold')
axes[2].set_xlabel('Thresholded magnitude'); axes[2].set_ylabel('Density')
axes[2].grid(True, alpha=0.3)

plt.suptitle('ADMM Sub-problem B: Proximal Soft-Thresholding Analysis', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'proximal_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Proximal operator analysis saved')

## 8. Save Fine-Tuning History & Phase 2B Summary

In [ ]:
# ── Save training history ─────────────────────────────────────────────────────
history_path = os.path.join(OUT_DIR, 'finetune_history.json')
with open(history_path, 'w') as f:
    json.dump(ft_history, f, indent=2)

# ── Final weights also saved in weights/ ──────────────────────────────────────
# (best epoch already saved in training loop)

print('\n' + '=' * 72)
print('📦  Phase 2B Outputs')
print(f'   Model weights : {FINAL_CKPT}')
print(f'   Train history : {history_path}')
print(f'   Hold-out split: {OUT_DIR}dataset2_split.json  → used in Notebook 04')
print(f'   Figures       : {OUT_DIR}finetune_convergence.png')
print('=' * 72)

print('\n📋 Phase 2B Summary')
print(f'   Final LNCC Loss      : {ft_history["lncc"][-1]:.4f}')
print(f'   Final TV Loss        : {ft_history["tv"][-1]:.4f}')
print(f'   Final Primal Residual: {ft_history["primal_res"][-1]:.5f}')
print(f'   Best Epoch           : {best_epoch}/{N_EPOCHS}')
print(f'   Best LNCC            : {best_loss:.4f}')
print(f'\n✅ cnn_lstm_final.pth is ready for Phase 3 statistical testing.')

## 9. Phase 2B Summary

| Component | Implementation | Mathematical Basis |
|-----------|---------------|--------------------|
| **Layer Freezing** | First 2 CNN encoder blocks frozen | Preserve low-level physics from Phase 2A |
| **Dataset** | Dataset 2 phantom/in-vivo `.mat` files (80% train) | Real transducer noise, real artifacts |
| **Label Usage** | None (fully unsupervised) | Label-free domain adaptation |
| **LNCC Loss** | Local NCC on consecutive displacement frames | Acoustic tracking fidelity |
| **TV Regularization** | Spatio-temporal Total Variation | Physical continuity (tissue doesn't tear) |
| **ADMM θ-step** | SGD on LNCC + consensus penalty | Smooth sub-problem |
| **ADMM z-step** | Soft-Thresholding with τ=λ/ρ | Non-smooth TV proximal operator |
| **ADMM u-step** | Dual ascent u ← u + (d − z) | Lagrange multiplier update |
| **Output** | `cnn_lstm_final.pth` | Production weights for clinical deployment |

**→ Proceed to `04_statistical_testing.ipynb` for Phase 3: Statistical Validation on hold-out phantom data.**